# Airport Cargo MAWM：Colab 快速闭环

这个 Notebook 会完成：安装 → 环境冒烟测试 → Oracle CEM → 采集小数据 → 训练状态世界模型 → 学习模型 CEM 评估。

这是状态空间 MVP，不是最终的像素 JEPA。正式实验前请先确保这里的每一格都能运行。

## 1. 克隆或更新代码

In [1]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/june250630/airport-cargo-mawm.git"
REPO_DIR = Path("/content/airport-cargo-mawm")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("工作目录：", Path.cwd())

工作目录： /content/airport-cargo-mawm


## 2. 安装依赖并检查 GPU

Colab 菜单中选择“代码执行程序 → 更改运行时类型 → GPU”。状态基线很小，CPU 也能训练；后续像素模型才真正需要 GPU。

In [2]:
!python -m pip install -q -e ".[dev]"
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.5/668.5 kB 2.3 MB/s eta 0:00:00
  Building editable for airport-cargo-mawm (pyproject.toml) ... done
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 3. 运行单元测试

In [3]:
!pytest

.....                                                                    [100%]
=============================== warnings summary ===============================
../../usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25
  /usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
    from pkg_resources import resource_stream, resource_exists

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
5 passed, 1 warning in 3.64s


## 4. 脚本专家冒烟测试

脚本专家读取真实仿真状态，只用于证明任务可解以及采集训练数据。

In [4]:
LOCAL_ROOT = Path("/content/acmawm_runs")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
!python -m airport_cargo_mawm.cli.smoke --episodes 3 --videos 1 --output /content/acmawm_runs/smoke
from IPython.display import Video, display
display(Video("/content/acmawm_runs/smoke/episode_000.mp4", embed=True))

{"episode": 0, "success": true, "steps": 15, "return": 6.506119331717491}
{"episode": 1, "success": true, "steps": 38, "return": 2.6928278152644634}
{"episode": 2, "success": true, "steps": 16, "return": 6.230889912098645}
success_rate=100.0%; output=/content/acmawm_runs/smoke


## 5. 真实动力学 Oracle CEM

Oracle使用特权全局状态和精确动力学，只用于判断CEM搜索是否能完成任务，不是可部署策略或数学最优上界。

In [ ]:
!python -m airport_cargo_mawm.cli.oracle --cost task --episodes 2 --seed 0 --search-seed 1000 --horizon 20 --samples 256 --elites 32 --iterations 4 --videos 1 --output /content/acmawm_runs/oracle_task
display(Video("/content/acmawm_runs/oracle_task/episode_000.mp4", embed=True))

## 6. 采集小数据并训练 5 个 epoch

这只是管线测试。正式实验应扩大场景数、使用独立保留集，并训练多个随机种子。

In [5]:
!python -m airport_cargo_mawm.cli.collect --episodes 20 --seed 0 --output /content/acmawm_runs/train.h5
!python -m airport_cargo_mawm.cli.train --data /content/acmawm_runs/train.h5 --output /content/acmawm_runs/checkpoint --epochs 5 --batch-size 128 --seed 0

{
  "path": "/content/acmawm_runs/train.h5",
  "episodes": 20,
  "transitions": 601,
  "success_rate": 0.95
}
{"epoch": 0, "train_loss": 0.0023749101568352094, "val_loss": 0.0010879881447181106}
{"epoch": 1, "train_loss": 0.0012864792161950808, "val_loss": 0.0009513974073342979}
{"epoch": 2, "train_loss": 0.0009848225127549381, "val_loss": 0.0007573776529170573}
{"epoch": 3, "train_loss": 0.0009196713058785958, "val_loss": 0.0007844906649552286}
{"epoch": 4, "train_loss": 0.0008207826875825727, "val_loss": 0.0007024806109257042}
checkpoint=/content/acmawm_runs/checkpoint/last.pt; device=cuda


## 7. 用学习模型进行小规模 CEM 评估

初始模型可能无法成功完成任务；这里首先验证规划代码能否执行并生成结果。

In [ ]:
!python -m airport_cargo_mawm.cli.evaluate --checkpoint /content/acmawm_runs/checkpoint/last.pt --cost task --episodes 2 --videos 1 --samples 64 --elites 8 --iterations 2 --output /content/acmawm_runs/eval
display(Video("/content/acmawm_runs/eval/episode_000.mp4", embed=True))

## 8. 持久化到 Google Drive

`/content` 会在运行时结束后被删除。确认前面全部成功后，再运行本格保存数据和权重。

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/airport_cargo_mawm/quickstart")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
print("已保存到：", DRIVE_ROOT)

## 下一步

1. 把数据扩展到至少 100 个训练场景和独立测试场景；
2. 训练 3 个不同随机种子的模型，组成模型集成；
3. 对比 `uncertainty_weight=0` 与大于 0 的校准规划；
4. 对比 Oracle 与学习模型在相同 task/shaped 代价下的差距；
5. 扩展到 VMAS 机场场景和像素 JEPA。